## Goal: Segment complex parts into segments that are bijective along their tertiary axis

In [1]:
import numpy as np
import open3d as o3d
from sklearn.cluster import KMeans
import matplotlib.cm as cm
import matplotlib.pyplot as plt

mesh_file= r"plane_segments\Airfoil_surface.stl"
mesh = o3d.io.read_triangle_mesh(mesh_file)
mesh.remove_duplicated_vertices()
print(f"Mesh loaded: {len(mesh.triangles)} triangles, {len(mesh.vertices)} vertices")
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Mesh loaded: 4958 triangles, 2576 vertices


## Normal-based k-means clustering

In [2]:
k_clusters=6

t_mesh=o3d.t.geometry.TriangleMesh.from_legacy(mesh)
t_mesh.compute_triangle_normals()
normals=t_mesh.triangle.normals.cpu().numpy()

triangle_colors = o3d.core.Tensor(np.random.rand(len(normals), 3), dtype=o3d.core.float32)
t_mesh.triangle.colors=triangle_colors
#o3d.visualization.draw([t_mesh])


kmeans= KMeans(n_clusters=k_clusters, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(normals)
print(f"Mesh loaded: {len(normals)} triangles")
print(f"Clustering complete for {k_clusters} segments")

# 5. Assign True Per-Triangle Colors (Tensor API)
cmap = cm.get_cmap('Spectral', k_clusters)
cluster_colors = np.array([cmap(i)[:3] for i in range(k_clusters)], dtype=np.float32)
face_colors_np = cluster_colors[cluster_labels]

# Convert NumPy array to Open3D Tensor and assign to triangle_colors
print(face_colors_np[0])
face_colors_tensor = o3d.core.Tensor(face_colors_np)
t_mesh.triangle.colors = face_colors_tensor

# 6. Visualize
o3d.visualization.draw_geometries([t_mesh.to_legacy()])



Mesh loaded: 4958 triangles
Clustering complete for 6 segments
[0.36862746 0.30980393 0.63529414]


C:\Users\jonas\AppData\Local\Temp\ipykernel_15480\607439076.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('Spectral', k_clusters)


## Watershed Segmentation Algorithm

In [3]:
from collections import defaultdict
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def compute_per_triangle_normals(mesh):
    #Calculates and returns the normalized normal vector of every triangle
    tri_verts=np.asarray(mesh.vertices)[np.asarray(mesh.triangles)]
    v0s= tri_verts[:,1] - tri_verts[:,0]
    v1s= tri_verts[:,2] - tri_verts[:,0]
    normals=np.cross(v0s,v1s)
    norms= np.linalg.norm(normals, axis=1, keepdims=True)
    normals= normals/norms
    return normals

def vertex_to_triangle_adjacency(mesh):
    #Stores the indicies of every triangle a vertex is a part of 
    n_verts= len(mesh.vertices)
    tri_adj=[[] for i in range(n_verts)]
    triangles = np.asarray(mesh.triangles)
    for index, tri in enumerate(triangles):
        for v in tri:
            tri_adj[v].append(index)
    return tri_adj

def build_vertex_adjacency(mesh):
    #Determines every neighboring vertices a vertex has by iterating through all triangles 
    #and adding the other two vertices to each other's neighbor sets
    n=len(mesh.vertices)
    adj=[set() for i in range(n)]
    triangles = np.asarray(mesh.triangles)
    for tri in triangles:
        a,b,c=tri
        adj[a].update([b,c])
        adj[b].update([a,c])
        adj[c].update([a,b])

    adj_list= [np.array(sorted(list(sett)), dtype=int) for sett in adj]
    return adj_list

def compute_vertex_curvature(mesh):
    #Calculates vertex curvature value from frobenous norm of 
    #covariance matrix of adjacent triangles
    tri_normals= compute_per_triangle_normals(mesh) #normalized triangle norm vectors
    tri_adj = vertex_to_triangle_adjacency(mesh)#returns list of all adjacent triangles for every vertex
    n=len(mesh.vertices)
    curvature=np.zeros(n, dtype=float)
    for vertex in range(n):
        tri_indx = tri_adj[vertex]
        if len(tri_indx)==0:
            curvature[vertex] =0
            print("Unconnected vertex detected!")
            continue
        neighbor_normals=tri_normals[tri_indx]

        if neighbor_normals.shape[0]==1:
            curvature[vertex]=0.0
            continue

        cov=np.cov(neighbor_normals.T)
        curvature[vertex]= np.linalg.norm(cov, ord='fro')

    return curvature


def find_local_minima(curv,adj):
    #takes vertex curvature values and highlights
    # vertices that are local minima
    n= len(curv)
    is_min=np.zeros(n, dtype=bool)
    for vertex in range(n):
        neigh=adj[vertex]
        if np.all(curv[vertex] < curv[neigh] - 1e-12):
            is_min[vertex]=True
    return is_min


def find_flat_components(curv, adj, flat_tol=1e-6):
    #Depth first search to find connected components
    n = len(curv)
    visited = np.zeros(n, dtype=bool)
    comp_id = -np.ones(n,dtype=int)
    components = []
    cid = 0 #component ID
    for vertex in range(n):
        if visited[vertex]:#if already considered, move on
            continue
        stack = [vertex] #functions as to-search list
        visited[vertex]=True #mark as considered
        comp =[] # add visited successful candidates for labeling
        is_real_patch = False #Ensures patches have at least 2 vertices
        while stack:
            u=stack.pop() #remove from to-search list
            comp.append(u) 
            for v in adj[u]: #compare neighbor vertices
                if not visited[v] and abs(curv[u]-curv[v]) <=flat_tol:
                    is_real_patch=True
                    visited[v]=True 
                    stack.append(v) #add flat neighbor to to-search list
        
        if is_real_patch: 
            components.append(np.array(comp,dtype=int)) #Should only have flat regions
            for v in comp:
                comp_id[v] = cid #assign all vertices in comp the same CID,single vertices remain as -1
            cid += 1
    return components, comp_id

def classify_and_label_flat_components(components, comp_id, curv, adj):
    #Determine if flat regions are plateaus or minima
    labels=-np.ones(len(curv), dtype=int)
    next_label=0
    comp_type = ["flat_min"] * len(components) #assume all are plateaus initially
    #print(len(curv), len(comp_id), len(components))
    for cid, comp in enumerate(components): 
        is_a_plateu=False
        for vertex in comp: #search all neighboring vertices outside of patch for lower curvature
            outside=np.array([u for u in adj[vertex] if comp_id[u]!=cid],dtype = int)
            if outside.size == 0: #Interior point with no outside neighbors
                continue
            if np.any(curv[outside] < curv[vertex] - 1e-12): 
                is_a_plateu = True #Downhill path found!
                break 
        
        if is_a_plateu:
            comp_type[cid] = "plateau" #change type to reflect downhill path
        else:
            for v in comp:
                labels[v] = next_label#only label the flat minima, since others will descend to them
            next_label += 1
    return labels, next_label, comp_type

def steepest_descent_path(start,curv,adj,labels):
    path=[] #list of vertices visited on journey
    seen = set() #set of visited vertices
    u = start #start vertex
    while True:
        path.append(u) #add start to path and set
        seen.add(u)
        if labels[u] != -1: # Sucdess, labeled basin found! 
            return path, labels[u]
        
        neigh = adj[u] #if not, collect all neighbors
        
        if neigh.size == 0:
            print(f"Mesh error: 0 neighbors detected for vertex {u}")
            return path, -2
        
        min_idx = int(neigh[np.argmin(curv[neigh])]) #return index of minimum curvature neighbor
        
        if curv[min_idx]>= curv[u] - 1e-15: #If current node is lower curvature than all neighbor nodes
            #print("New label needed")
            return path, -1 #if there's an unlabeled local minima 
        
        u = min_idx #now look at the next point
        
        if u in seen:
            print("Mesh error: Path from {start} created cycle at {u}. Skipping path")
            return path, -2
        
def descend_plateaus(components, comp_id, comp_type, curv, adj, labels, next_label):
    #descent plateaus until either existing minima or new minima is discovered
    for cid, comp in enumerate(components): 
        if comp_type[cid]=="flat_min": #skips flat_min's becuase they already have a label
            continue 
        if np.any(labels[comp]!=-1): #skips components that have already labeled portions
            continue
        boundary=[]
        for v in comp: #collect outside neighbors
            for nb in adj[v]: 
                if comp_id[nb] !=cid:
                    boundary.append(nb)

        if len(boundary) == 0:
            print(f"Mesh Warning: 0 neighbors detected for patch {cid}")
            newlabel = next_label
            next_label += 1
            labels[comp] = newlabel
            continue

        boundary = np.array(boundary,dtype=int)
        seed = int(boundary[np.argmin(curv[boundary])]) #give "spillover" vertex
        path, encountered_label = steepest_descent_path(seed,curv,adj,labels) #follow steepest descent from seed vertex
        if encountered_label == -1:
            #Found new, unlabeled minimum!
            encountered_label = next_label
            next_label += 1

        labels[comp] = encountered_label #assign discovered label to entire patch
        for v in path:
            labels[v] = encountered_label #assign discovered label to entire descent path too
    
    return labels, next_label


def descend_remaining_vertices(curv,adj,labels, next_label):
    #Final check of un-labeled vertices
    n=len(curv)
    for i in range(n):
        if labels[i] != -1: 
            continue

        path, encountered_label = steepest_descent_path(i,curv,adj,labels)
        if encountered_label == -1: 
            #found new, unlabeled minimum!
            encountered_label = next_label
            next_label+=1

        for v in path:
            labels[v] = encountered_label #assign discovered label for entire descent path

    return labels, next_label
    
def compute_region_boundary_vertices(labels, adj):
    #find the ridgeline of each labeled region
    region_vertices = defaultdict(list)
    for v_idx,label in enumerate(labels):
        region_vertices[label].append(v_idx) #create dictionary of each labeled region
    boundaries={}
    for label, verts in region_vertices.items():#check for vertices that border other labeled regions
        border_vs = set()
        for v in verts:
            for nb in adj[v]:
                if labels[nb]!= label:
                    border_vs.add(v)#add all boundary vertices to list
                    break
        boundaries[label]=np.array(sorted(list(border_vs)),dtype=int) #Final dictionary of all boundary vertices for each labeled region
    return boundaries

def compute_watershed_depths(curv,labels,adj):
    #Calculated how "deep" each basin region is
    unique_labels = np.unique(labels)
    depths={}
    boundaries = compute_region_boundary_vertices(labels, adj)
    for lab in unique_labels: #iterater through each labeled region
        lab = int(lab)
        region_vs = np.where(labels==lab)[0] #get all vertices in labeled region
        region_min = curv[region_vs].min() #find minimum curvature in region
        border_vs = boundaries.get(lab, np.array([],dtype=int))
        if border_vs.size == 0:
            depths[lab]=np.inf
            
        else:
            border_min = curv[border_vs].min() #minimum curvature on boundary
            depths[lab] = border_min - region_min #Final region depth, smallest "ridge-to-basin" measurement

    return depths, boundaries

def merge_shallow_regions(labels, curv, adj, depththresh):
    #"De-noises" regions combining shallow regions
    labels = labels.copy()
    while True:
        depths, boundaries = compute_watershed_depths(curv, labels, adj)
        shallow = [lab for lab, d in depths.items() if d < depththresh] # Regions too shallow
        if len(shallow) == 0:
            break #no shallow regions to merge
        merged_any = False
        for lab in shallow:
            border_vs = boundaries.get(lab, np.array([], dtype=int)) #all border points of region I
            if border_vs.size == 0:
                continue
            b = int(border_vs[np.argmin(curv[border_vs])]) #smallest curvature on boundary
            neighbor_regions = set(labels[nb] for nb in adj[b] if labels[nb] != lab) #set of regions neighboring shallow region I
            if len(neighbor_regions) == 0:
                continue
            best_neighbor = None
            best_curv = np.inf
            for nbr in neighbor_regions:
                shared = [nb for nb in adj[b] if labels[nb] == nbr] #collect neighbors of smallest boundary vertex for each neighboring label
                if len(shared) == 0:
                    continue
                c = curv[np.array(shared)].min()
                if c < best_curv:
                    best_curv = c
                    best_neighbor = nbr
            if best_neighbor is None:
                continue
            labels[labels == lab] = best_neighbor #Merge all of shallow region I into neighboring region with next lowest curvature
            merged_any = True
        if not merged_any:
            break
    # compress labels into consecutive final labels
    unique = np.unique(labels)
    mapping = {old:i for i, old in enumerate(unique)}
    for old, new in mapping.items():
        labels[labels == old] = new
    return labels

# Full run pipeline
def mesh_watershed(mesh, flat_tol=1e-8, depththresh=0.001):

    mesh.compute_vertex_normals()
    curv = compute_vertex_curvature(mesh)
    print(np.mean(curv), np.median(curv), np.max(curv))
    adj = build_vertex_adjacency(mesh)
    is_min = find_local_minima(curv, adj)

    #-------------Display curvature for visualization-------------
    curv_min=curv.min()
    curv_max=curv.max()
    curv_norm = (curv - curv_min)/ (curv_max - curv_min)
    colors = plt.cm.jet(curv_norm)[:,:3]
    mesh.vertex_colors= o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([mesh],window_name="Curvature Values", mesh_show_back_face=True)
    #-------------------------------------------------------------
    
    components, comp_id = find_flat_components(curv, adj, flat_tol=flat_tol)
    labels, next_label, comp_type = classify_and_label_flat_components(components, comp_id, curv, adj)
    labels, next_label = descend_plateaus(components, comp_id, comp_type, curv, adj, labels, next_label)
    labels_before_merging, next_label  = descend_remaining_vertices(curv, adj, labels.copy(), next_label)

    depths = compute_watershed_depths(curv, labels_before_merging, adj)
    merged_labels = merge_shallow_regions(labels_before_merging, curv, adj, depththresh)
    
    unique_after = np.unique(merged_labels)
    cmap = np.random.RandomState(50).rand(len(unique_after), 3)
    colors = cmap[merged_labels % len(unique_after)]
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([mesh], window_name="Final Groupings", mesh_show_back_face=True)
    
    return {
        "curvature": curv,
        "is_min": is_min,
        "labels_before_merge": labels_before_merging,
        "labels_after_merge": merged_labels,
        "depths": depths,
        "components": components,
        "comp_id": comp_id,
        "comp_type": comp_type,
        "mesh": mesh
    }

def demo_and_save(mesh=None):
    if mesh== None:
        mesh = o3d.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=80)
        vertices = np.asarray(mesh.vertices)
        #mesh = o3d.geometry.TriangleMesh.create_box(width=5)
        noise = 0.05 * np.sin(10 * vertices[:,0]) * np.sin(8 * vertices[:,1])
        vertices_noisy = vertices.copy()
        vertices_noisy[:,2] += noise
        mesh.vertices = o3d.utility.Vector3dVector(vertices_noisy)
    
    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    vertices = np.asarray(mesh.vertices)
    # add patterned bump for curvature features
    mesh.compute_vertex_normals()

    # We need a clean copy of the mesh for our visualizations
    base_mesh = o3d.geometry.TriangleMesh(mesh) 

    # Run the full pipeline
    res = mesh_watershed(mesh, flat_tol=1e-8, depththresh=0.0005)
    print("Run complete. Starting visualizations...")

    # # Visualize Local Minima
    # minima_mesh = o3d.geometry.TriangleMesh(base_mesh)
    # is_min = res['is_min']
    # minima_colors = np.full((len(base_mesh.vertices), 3), 0.5) # Grey
    # minima_colors[is_min] = [1.0, 0.0, 0.0] # Red
    # minima_mesh.vertex_colors = o3d.utility.Vector3dVector(minima_colors)
    # o3d.visualization.draw_geometries([minima_mesh], window_name="Stage 1: Local Minima")

    # # Visualize Plateaus  
    # print("Displaying Stage 2: Flat Components. Close window to continue.")
    # flat_mesh = o3d.geometry.TriangleMesh(base_mesh) # Use a fresh copy
    # components = res['components']
    # flat_colors = np.full((len(base_mesh.vertices), 3), 0.5) # Grey
    # comp_cmap = np.random.RandomState(12).rand(len(components), 3)
    # for i, comp_indices in enumerate(components):
    #     if len(comp_indices) > 1: # Only color non-trivial components
    #         flat_colors[comp_indices] = comp_cmap[i]
            
    # flat_mesh.vertex_colors = o3d.utility.Vector3dVector(flat_colors)
    # o3d.visualization.draw_geometries([flat_mesh], window_name="Stage 2: Flat Components")

    # Visualize all basins before merging step
    # print("Displaying Stage 3: Unmerged Basins. Close window to exit.")
    # unmerged_mesh = o3d.geometry.TriangleMesh(base_mesh) # Use a fresh copy
    # labels_before = res['labels_before_merge']
    # unique_before = np.unique(labels_before)
    
    # cmap_before = np.random.RandomState(42).rand(len(unique_before), 3)
    # colors_before = cmap_before[labels_before % len(unique_before)]
    
    # unmerged_mesh.vertex_colors = o3d.utility.Vector3dVector(colors_before)
    # o3d.visualization.draw_geometries([unmerged_mesh], window_name="Stage 3: Unmerged Basins")

    # print("All stages complete.")
    # return res

if __name__ == "__main__":
    mesh = o3d.io.read_triangle_mesh("mesh_bodies/Joystick.stl")
    #mesh=o3d.io.read_triangle_mesh("plane_segments\Airfoil_Surface_example_Hypermeshed.stl")
    res = demo_and_save(mesh)


0.005265943213258596 0.006107128265283003 0.010162549512411936
Run complete. Starting visualizations...


## Surface-Normal Deviation Segmentation Algorithm

In [4]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

def build_triangle_adjacency(mesh):
    
    mesh.compute_vertex_normals()
    n_tris = len(mesh.triangles)
    vert_to_tri_list=[[] for i in range(len(mesh.vertices))]
    triangles = np.asarray(mesh.triangles)
    for t_idx, tri in enumerate(triangles):
        for v in tri:
            vert_to_tri_list[v].append(t_idx) #add triangle to vertex's list

    tri_adj = [set() for j in range(n_tris)]
    for tri_idx, tri in enumerate(triangles):
        for v in tri: #add neighboring triangle to triangle's adjacency set
            tri_adj[tri_idx].update(vert_to_tri_list[v])
        tri_adj[tri_idx].discard(tri_idx)

    return [list(s) for s in tri_adj]

def segment_by_normal_deviation(mesh, max_angle_dev=60):
    print(f"Segmenting Mesh with maximum deviation across patch of {max_angle_dev} degrees")
    
    #Build Adjacency
    mesh.compute_triangle_normals()
    tri_normals=np.asarray(mesh.triangle_normals)
    triangles=np.asarray(mesh.triangles)
    n_tris= len(triangles)

    #Flatness Score
    tri_adj = build_triangle_adjacency(mesh)
    flatness_scores=np.zeros(n_tris, dtype=float)
    for i in range(n_tris):
        neighbors= tri_adj[i]
        if not neighbors:
            flatness_scores[i]=1.0
            continue

        nb_normals=tri_normals[neighbors]
        dots=np.dot(nb_normals, tri_normals[i])
        flatness_scores[i]= np.min(dots)

    sorted_indices = np.argsort(-flatness_scores) 

    #Region Growing
    labels= -np.ones(n_tris, dtype=int)
    current_label=0
    threshold=np.cos(np.deg2rad(max_angle_dev))
    for seed_idx in sorted_indices:
        if labels[seed_idx] != -1:
            continue #already labeled
        if flatness_scores[seed_idx] < 0.2:
            labels[seed_idx] = -2 #mark as noise
            continue
        labels[seed_idx]=current_label
        seed_normal= tri_normals[seed_idx]
        queue=[seed_idx]
        while queue:
            curr_tri = queue.pop(0)
            neighbors = tri_adj[curr_tri]
            for nb in neighbors:
                if labels[nb] != -1:
                    continue #already labeled
                deviation = np.dot(tri_normals[nb], seed_normal)
                if deviation >= threshold:
                    labels[nb]=current_label
                    queue.append(nb)

        current_label += 1
    print(f"Segmentation complete: {current_label} segments found.")
    return labels

def visualize_triangle_labels(mesh, labels):
    max_label = np.max(labels)
    if max_label < 0:
        print("No segments to visualize.")
        return
    cmap= plt.get_cmap("tab10")
    tri_colors = np.zeros((len(labels),3))
    for i, label in enumerate(labels):
        if label == -2:
            tri_colors[i] = [0.2,0.2,0.2] #Gray for noise
        else:
            color = cmap(label % 10)[:3]
            tri_colors[i] = color

    vertex_colors=np.zeros((len(mesh.vertices),3))
    vertex_count=np.zeros(len(mesh.vertices))
    triangles=np.asarray(mesh.triangles)
    for tri_idx, tri in enumerate(triangles):
        for v in tri:
            vertex_colors[v] += tri_colors[tri_idx]
            vertex_count[v] += 1
    mask = vertex_count > 0
    vertex_colors[mask] /= vertex_count[mask][:, None]
    vis_mesh = o3d.geometry.TriangleMesh(mesh)
    vis_mesh.vertex_colors = o3d.utility.Vector3dVector(vertex_colors)
    o3d.visualization.draw_geometries([vis_mesh], window_name="Segmented Mesh", mesh_show_back_face=True)


def visualize_one_by_one(mesh, labels):
    #Visualize each segment individually
    unique_labels = np.unique(labels)
    unique_labels = unique_labels[unique_labels >= 0]
    triangles = np.asarray(mesh.triangles)
    vertices = np.asarray(mesh.vertices)

    base_mesh = o3d.geometry.TriangleMesh(mesh)
    base_mesh.vertices = o3d.utility.Vector3dVector(vertices)
    
    for label in unique_labels:
        group_tri_indices = np.where(labels == label)[0]
        sub_mesh = o3d.geometry.TriangleMesh(base_mesh)
        remove_mask = labels != label
        sub_mesh.remove_triangles_by_mask(remove_mask)
        sub_mesh.remove_unreferenced_vertices()
        sub_mesh.paint_uniform_color([1, 0.1, 0.1])
        sub_mesh.compute_vertex_normals()
        line_set = o3d.geometry.LineSet.create_from_triangle_mesh(sub_mesh)
        line_set.paint_uniform_color([0, 0, 0]) # Black wireframe
        print(f"Displaying Group {label} ({len(group_tri_indices)} triangles)...")
        o3d.visualization.draw_geometries([sub_mesh, line_set], 
                                          window_name=f"Group {label} (Close to Next)")

def prune_small_segments(mesh, labels, min_triangles=5):
    # Count triangles makeing up each segment
    unique, counts = np.unique(labels, return_counts=True)
    small_labels = set(unique[counts < min_triangles])
    
    if not small_labels:
        return labels # Nothing to fix
    print(f"Pruning {len(small_labels)} tiny region(s)...")
    
    # Iterate only through triangles that belong to small groups
    tri_adj = build_triangle_adjacency(mesh) 
    new_labels = labels.copy()
    for i in range(len(labels)):
        if new_labels[i] in small_labels:
            my_neighbors = tri_adj[i]
            neighbor_labels = [new_labels[n] for n in my_neighbors if new_labels[n] != -1]
            valid_neighbors = [l for l in neighbor_labels if l not in small_labels]
            if valid_neighbors:
                # Pick the most frequent valid neighbor label
                best_neighbor = max(set(valid_neighbors), key=valid_neighbors.count)
                new_labels[i] = best_neighbor
    return new_labels


if __name__ == "__main__":
    mesh_file= r"plane_segments\Airfoil_surface.stl"
    mesh_file= r"mesh_bodies/Joystick.stl"
    mesh = o3d.io.read_triangle_mesh(mesh_file)
    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.remove_degenerate_triangles()
    print(f"Mesh loaded: {len(mesh.triangles)} triangles, {len(mesh.vertices)} vertices")
    labels = segment_by_normal_deviation(mesh, max_angle_dev=45)
    labels= prune_small_segments(mesh, labels, min_triangles=20)
    visualize_triangle_labels(mesh, labels)
    #visualize_one_by_one(mesh, labels)


Mesh loaded: 7462 triangles, 3733 vertices
Segmenting Mesh with maximum deviation across patch of 45 degrees
Segmentation complete: 36 segments found.
Pruning 11 tiny region(s)...


## Bin-Based Morse/Boustrophedon Decomposition

In [5]:
import open3d as o3d
import numpy as np


def segment_boustrophedon_pca(mesh, bin_size=20, min_triangle_count=5):
    #PCA
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    centroid=np.mean(vertices, axis=0)
    centered_verts= vertices-centroid
    cov= np.cov(centered_verts.T)
    eigvals, eigvecs = np.linalg.eigh(cov) #PCA Calculations
    principal_axis = eigvecs[:,-1]
    print(eigvecs,eigvals, f"Principal Axis: {principal_axis}")
    
    #Slice based on triangles, not vertices
    tri_centroids= np.mean(vertices[triangles], axis=1)

    # Scan Height along Principal Component
    projected_coords = np.dot(tri_centroids, principal_axis) #gives "height" value along principal axis
    min_val, max_val = np.min(projected_coords), np.max(projected_coords)
    range_val= max_val-min_val
    num_bins = int(range_val/bin_size)
    if num_bins<5:
        num_bins, true_bin_size = 5, range_val/5
        print(f"Bin size too large, adjusted bin size to {true_bin_size}mm")
    bins=np.linspace(np.min(projected_coords), np.max(projected_coords), num_bins+1)
    critical_heights= [min_val]
    previous_num_components = -1
    
    # Scan for critical points
    print(f"scanning {num_bins} slices for connectivity changes")
    for b in range(len(bins)-1):
        lower, upper= bins[b], bins[b+1]
        mask_tri_indices = (projected_coords>=lower) & (projected_coords<= upper)
        mask_indices=np.where(mask_tri_indices)[0] 
        if len(mask_tri_indices)==0:
            current_num_components = 0
        else:
            sub_mesh=mesh.select_by_index(np.unique(triangles[mask_indices])) 
            #o3d.visualization.draw_geometries([sub_mesh], mesh_show_back_face=True)
            _, cluster_sizes, _ = sub_mesh.cluster_connected_triangles()
            real_clusters=[s for s in cluster_sizes if s> min_triangle_count] #filter out triangles "cut-off" by strict binning
            current_num_components = len(real_clusters)
            if previous_num_components != -1 and current_num_components != previous_num_components:
                if current_num_components >0 and previous_num_components >0:
                    print(f"  -> Change at {lower:.1f} from {previous_num_components} to {current_num_components} connected parts")
                    critical_heights.append(lower)
        previous_num_components=current_num_components
    
    # Label initial bands
    critical_heights.append(max_val)
    print(f"Found {len(critical_heights)-1} topological bands")
    final_vertex_labels = np.zeros(len(vertices))
    current_region_id=0

    for i in range(len(critical_heights)-1):
        lower, upper =critical_heights[i], critical_heights[i+1]
        band_mask = (projected_coords >= lower - 1e-6) & (projected_coords <= upper + 1e-6)
        sub_mesh=mesh.select_by_index(np.unique(triangles[band_mask])) #these mesh vertices have original index values?
        global_tri_indices = np.where(band_mask)[0]       
        subset_triangles= triangles[global_tri_indices]
        subset_unique_vertices=np.unique(subset_triangles) #same as np.unique(triangles[band_mask])

        #Cluster Submesh
        tri_clusters, cluster_sizes,_ = sub_mesh.cluster_connected_triangles()
        tri_clusters = np.asarray(tri_clusters)

        sub_triangles= np.asarray(sub_mesh.triangles)
        flat_local_verts=sub_triangles.flatten()
        flat_cluster_ids=np.repeat(tri_clusters,3)
        flat_global_verts= subset_unique_vertices[flat_local_verts]
        final_vertex_labels[flat_global_verts]= current_region_id + flat_cluster_ids

        num_clusters_in_band=0
        if len(cluster_sizes)>0:
            num_clusters_in_band = len(cluster_sizes)
        current_region_id += num_clusters_in_band
    
    print(f"Final Result: {current_region_id} PCA-Aligned Regions")
    return final_vertex_labels, principal_axis, centroid


def visualize_pca_segmentation(mesh, labels, principal_axis, centroid):
    #Color Regions
    max_label=round(labels.max())
    colors = np.random.rand(max_label+1,3)
    vertex_colors=np.zeros((len(mesh.vertices),3))
    for i in range(len(labels)):
        vertex_colors[i]=colors[round(labels[i])]
    mesh.vertex_colors=o3d.utility.Vector3dVector(vertex_colors)
    
    #Show Principal Component
    arrow=o3d.geometry.TriangleMesh.create_arrow()
    default_axis = np.array([0,0,1])
    a, b = default_axis, principal_axis
    v = np.cross(a, b)
    c = np.dot(a, b)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    R = np.eye(3) + vx + np.dot(vx, vx) * (1 / (1 + c)) # Skew-symmetric matrix for rotation formula
    arrow.rotate(R, center=(0,0,0))
    arrow.translate(centroid)
    arrow.paint_uniform_color([1, 0, 0])
    print("Visualizing... (Close window to exit)")
    o3d.visualization.draw_geometries([mesh, arrow], window_name="PCA Boustrophedon Segmentation", mesh_show_back_face=True)

def debug_visualize_bands(mesh, labels):
    unique_labels = np.unique(labels)
    print(f"--- Debugging {len(unique_labels)} Regions ---")
    mesh.compute_vertex_normals()
    
    for label in unique_labels:
        indices = np.where(labels == label)[0]
        if len(indices) == 0: continue
            
        # Create sub-mesh
        sub_mesh = mesh.select_by_index(indices)
        sub_mesh.paint_uniform_color([1, 0.1, 0.1])
        line_set = o3d.geometry.LineSet.create_from_triangle_mesh(sub_mesh)
        line_set.paint_uniform_color([0, 0, 0])
        center = sub_mesh.get_center()
        print(f"Region {label}: {len(sub_mesh.triangles)} triangles. Center: {center}")
        bbox = mesh.get_axis_aligned_bounding_box()
        bbox.color = (0.5, 0.5, 0.5)
        o3d.visualization.draw_geometries([sub_mesh, line_set, bbox], 
                                          window_name=f"Region {label} (Press Q)")



#Depends on bins, or spacial bands, used to detect changes in connectivity
#Big difference between 2mm bins and 40mm bins in accurate calculation of critical points
if __name__ == "__main__":
    mesh_file= r"plane_segments\Airfoil_surface.stl"
    #mesh= o3d.io.read_triangle_mesh(mesh_file)
    mesh = o3d.geometry.TriangleMesh.create_torus(torus_radius=100.0, tube_radius=30.0, radial_resolution=600, tubular_resolution=200)
    R = mesh.get_rotation_matrix_from_xyz((np.pi/4, np.pi/4, 0))
    mesh.rotate(R, center=(0,0,0))
    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    mesh.compute_vertex_normals()
    labels, axis, center = segment_boustrophedon_pca(mesh, bin_size=40)
    #debug_visualize_bands(mesh, labels)
    visualize_pca_segmentation(mesh, labels, axis, center)


[[ 0.70710678 -0.27410674  0.65181707]
 [-0.5        -0.84563981  0.18679753]
 [ 0.5        -0.45799433 -0.73501101]] [ 450.00375003 5225.04354203 5225.04354203] Principal Axis: [ 0.65181707  0.18679753 -0.73501101]
scanning 6 slices for connectivity changes
  -> Change at -43.3 from 1 to 2 connected parts
  -> Change at 43.3 from 2 to 1 connected parts
Found 3 topological bands
Final Result: 4 PCA-Aligned Regions
Visualizing... (Close window to exit)


## Alternative Morse/Boustrophedon Decomposition

In [6]:
import open3d as o3d
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt


def build_dual_graph(mesh):
    #Builds adjacency list: adj[tri_id] = {neighbor_tri_ids}
    triangles = np.asarray(mesh.triangles)
    edge_to_tri = defaultdict(list)
    
    for idx, tri in enumerate(triangles):
        for e in [tuple(sorted((tri[0], tri[1]))), tuple(sorted((tri[1], tri[2]))), tuple(sorted((tri[2], tri[0])))]:
            edge_to_tri[e].append(idx) #store the triangles each edge is a part of
            
    tri_adj = [set() for _ in range(len(triangles))]
    for neighbors in edge_to_tri.values():
        if len(neighbors) == 2: #add edge-bordering triangles to each-others adjacency graph
            tri_adj[neighbors[0]].add(neighbors[1]) #neighbors are triangle indices
            tri_adj[neighbors[1]].add(neighbors[0])
    return tri_adj 

def color_dual_graph(num_triangles, tri_adj):
    #Greedily colors the dual graph with 2 colors (0 and 1) so that  neighbors rarely share a color. This allows for stable Red-Black updates.
    colors = np.full(num_triangles, -1, dtype=int)
    
    # If a triangle's neighbors are all uncolored, we pick 0. Elif a neighbor is 0, we pick 1.
    for t in range(num_triangles):
        neighbor_colors = {colors[n] for n in tri_adj[t] if colors[n] != -1}
        if 0 not in neighbor_colors:
            colors[t] = 0
        else:
            colors[t] = 1
            
    return colors

def get_connected_components(neighbors,edges):
    # Counts number of disjoint groups in neighbors verticies 
    if not neighbors: return 0

    adj=defaultdict(list)
    for n1,n2 in edges:
        if n1 in neighbors and n2 in neighbors:
            adj[n1].append(n2)
            adj[n2].append(n1)

    components = 0
    visited= set()
    #Breadth-First Search of neighboring vertices
    for n in neighbors:
        if n not in visited:
            components +=1
            queue = [n]
            visited.add(n)
            while queue:
                curr=queue.pop(0)
                for next_n in adj[curr]:
                    if next_n not in visited:
                        visited.add(next_n)
                        queue.append(next_n)

    return components

def find_critical_points(mesh, principal_axis):
    #Scans the mesh to find tolopogical saddles (splits/merges)
    vertices=np.asarray(mesh.vertices)
    triangles= np.asarray(mesh.triangles)
    heights= np.dot(vertices, principal_axis) #project onto princial component to get height

    linked_edges=defaultdict(list)
    for tri in triangles: #build neighbor map of connected vertices
        for i in range(3): 
            v= tri[i]
            n1= tri[(i+1)%3]
            n2= tri[(i+2)%3]
            linked_edges[v].append((n1,n2)) 
            #linked_edges is dictionary with each vertex as key and all neighboring vertices as values

    #Find critical points, via connectivity of lower neighbors
    saddle_heights=[]
    saddle_indices=[]

    for v_idx, edges in linked_edges.items(): #v_idx is vertex index, edges is list containing entries with the two other vertices for each neighboring triangle
        v_h=heights[v_idx]
        lower_neighbors = {n for edge in edges for n in edge if heights[n] < v_h} # assign neighbors as either upper or lower
        upper_neighbors = {n for edge in edges for n in edge if heights[n] > v_h} 
        for i in lower_neighbors:
            if abs(heights[i]-heights[v_idx])<1e-4:
                print("close!")
        for i in upper_neighbors:
            if abs(heights[i]-heights[v_idx])<1e-4:
                print("close!")
        is_saddle=False
        #Two separated components indicate a split/join happened
        if len(lower_neighbors)>=2:
            if get_connected_components(lower_neighbors, edges) >=2:
                is_saddle=True
        
        if not is_saddle and len(upper_neighbors)>=2:
            if get_connected_components(upper_neighbors,edges) >=2:
                is_saddle = True

        if is_saddle:
            saddle_heights.append(v_h)
            saddle_indices.append(v_idx)

    unique_saddles = sorted(list(set(saddle_heights)))
    print(f"Found {len(unique_saddles)} saddle points (Splits & Merges).")
    return unique_saddles, saddle_indices

def visualize_saddle_points(mesh, saddle_indices):
    #Colors the mesh dark gray and highlights saddle vertices (and neighbors) in Green.
    print(f"Visualizing {len(saddle_indices)} saddle points...")
    mesh.paint_uniform_color([0.2, 0.2, 0.2])
    mesh.compute_adjacency_list()
    adj_list = mesh.adjacency_list
    colors = np.asarray(mesh.vertex_colors)
    neon_green = [0, 1, 0]
    for idx in saddle_indices:
        # Paint the saddle itself
        colors[idx] = neon_green
        # Paint neighbors to make the dot visible
        for n in adj_list[idx]:
            colors[n] = [1,0,0]
            
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True, window_name="Detected Saddle Points (Green)")

def snap_saddle_ridges(mesh, saddle_indices, principal_axis, tolerance=1e-4):
    # Align the heights of vertices on a "degenerate" flat edge perpendicular (or close to) the principal axis
    vertices = np.asarray(mesh.vertices)
    heights = np.dot(vertices, principal_axis)
    aligned_heights=heights.copy()
    mesh.compute_adjacency_list()
    adj_list=mesh.adjacency_list
    print(f"Snapping flat ridges for {len(saddle_indices)} saddles")
    visited=set()
    for saddle_idx in saddle_indices:
        if saddle_idx in visited: 
            continue
        target_h = aligned_heights[saddle_idx]
        #Breadth First Search to find connected vertices at roughly similar height
        queue=[saddle_idx]
        visited.add(saddle_idx)
        ridge_indices=[saddle_idx]
        while queue:
            curr=queue.pop(0)
            for n in adj_list[curr]:
                if n in visited: 
                    continue
                #check if neighbor is "flat" relative to current vertex height
                if abs(aligned_heights[n]-target_h) < tolerance:
                    # is so, add index to the ridge, and queue up for exploration next
                    visited.add(n)
                    queue.append(n)
                    ridge_indices.append(n)
        if len(ridge_indices)>1:
            print(f"Saddle {saddle_idx} has {len(ridge_indices)-1} points affected by snapping")
            aligned_heights[ridge_indices]=target_h

    return aligned_heights

def segment_by_centroids(mesh, saddle_heights, aligned_heights, min_cluster_size=5):
    vertices=np.asarray(mesh.vertices)
    triangles=np.asarray(mesh.triangles)

    tri_heights=np.mean(aligned_heights[triangles], axis=1)
    cuts= [aligned_heights.min()]+saddle_heights+[aligned_heights.max()]
    cuts = sorted(list(set([round(c, 3) for c in cuts])))

    final_labels = np.zeros(len(vertices),dtype=int)
    region_counter=0
    print(f"Segmenting (centroid mode) into {len(cuts)-1} bands")
    
    for i in range(len(cuts)-1):
        lower,upper=cuts[i],cuts[i+1]
        if i==len(cuts)-2: #Last cut should include end
            mask_tri_indices = np.where((tri_heights >= lower - 1e-6) & (tri_heights<= upper + 1e-6))[0]
        else:
            
            mask_tri_indices = np.where((tri_heights >= lower - 1e-6) & (tri_heights < upper + 1e-6))[0]

        if len(mask_tri_indices)==0:
            print("Empty Slice Error")
            continue

        #Build Submesh from these triangles
        subset_triangles = triangles[mask_tri_indices]
        unique_v_indices = np.unique(subset_triangles)
        
        global_to_local = {global_id: local_id for local_id, global_id in enumerate(unique_v_indices)}
        vector_map = np.vectorize(global_to_local.get)
        local_triangles = vector_map(subset_triangles)
        sub_mesh = o3d.geometry.TriangleMesh()
        sub_mesh.vertices = o3d.utility.Vector3dVector(vertices[unique_v_indices])
        sub_mesh.triangles = o3d.utility.Vector3iVector(local_triangles)
        
        #Visualize
        sub_mesh.paint_uniform_color([1, 0.2, 0.6])
        wireframe = o3d.geometry.LineSet.create_from_triangle_mesh(mesh)
        wireframe.paint_uniform_color([0.8, 0.8, 0.8])
        o3d.visualization.draw_geometries([sub_mesh, wireframe], window_name="Triangle_Centroid_Clusters", mesh_show_back_face=True)
        
        # 4. Cluster and label "microclusters" 
        tri_labels, cluster_sizes, _ = sub_mesh.cluster_connected_triangles()
        tri_labels = np.asarray(tri_labels)
        if len(cluster_sizes) == 0: continue
        valid_cluster_map={}
        local_region_count=0
        for cid, size in enumerate(cluster_sizes):
            if size >= min_cluster_size:
                valid_cluster_map[cid] = region_counter + local_region_count
                local_region_count += 1
            else:
                valid_cluster_map[cid] = -1       

        # Map Back- iterate triangles to assign vertex labels
        # Note: Vertices on the cut boundary will be overwritten by the next band. This is expected behavior for vertex-painting.
        for t_idx, label in enumerate(tri_labels):
            if label == -1: continue
            
            global_region_id = valid_cluster_map[label]
            region_id = region_counter + label
            if global_region_id != -1:
                global_tri=subset_triangles[t_idx]
                final_labels[global_tri[0]] = global_region_id
                final_labels[global_tri[1]] = global_region_id
                final_labels[global_tri[2]] = global_region_id

        region_counter += local_region_count

    # "Heal" microclusters
    mesh.compute_adjacency_list()
    adj_list = mesh.adjacency_list
    
    # Iterative hole filling
    for _ in range(3): 
        fill_count = 0
        new_labels = final_labels.copy()
        for v_idx in range(len(vertices)):
            if final_labels[v_idx] == -1:
                # Look at neighbors
                neighbors = adj_list[v_idx]
                valid_neighbors = [final_labels[n] for n in neighbors if final_labels[n] != -1]
                
                if len(valid_neighbors) > 0:
                    # Take most common valid neighbor
                    counts = np.bincount(valid_neighbors)
                    new_labels[v_idx] = np.argmax(counts)
                    fill_count += 1
        
        final_labels = new_labels
        if fill_count == 0: break

    print(f"Centroid Segmentation: {region_counter} distinct regions.")
    return final_labels
        
def segment_by_saddles(mesh, saddle_heights, new_heights):
    # Segments mesh using exact saddle heights as boundaries.
    triangles = np.asarray(mesh.triangles)
    vertices = np.asarray(mesh.vertices) #mesh-wide vertex list
    
    # Prepare cutting points
    min_h, max_h = new_heights.min(), new_heights.max()
    cuts = [min_h] + saddle_heights + [max_h] #list of cut points
    cuts = sorted(list(set([round(c, 5) for c in cuts]))) #ordered list of cut points
    
    final_labels = np.zeros(len(vertices), dtype=int)
    region_counter = 0
    
    for i in range(len(cuts) - 1):
        lower, upper = cuts[i], cuts[i+1]
        mask_indices = np.where((new_heights >= lower - 1e-6) & (new_heights <= upper + 1e-6))[0] #mask all vertices in band (plus margin)
        if len(mask_indices) == 0: continue
        
        sub_mesh = mesh.select_by_index(mask_indices, cleanup=False) #o3d function select_by_index returns submesh (with new index scheme)
        # sub_mesh_dirty = mesh.select_by_index(mask_indices, cleanup=False)

        # clean_tree=o3d.geometry.KDTreeFlann(sub_mesh)
        # dirty_verts=np.asarray(sub_mesh_dirty.vertices)
        # colors=np.zeros_like(dirty_verts)+[0.8,0.8,0.8]
        # deleted_count=0
        # for v_i, point in enumerate(dirty_verts):
        #     # Search for this point in the clean mesh, radius=0.0001 (must be practically identical)
        #     k, _, _ = clean_tree.search_radius_vector_3d(point, 1e-3)
        #     # If k=0, the point was NOT found in the clean mesh -> It was deleted
        #     if k == 0:
        #         colors[v_i] = [1, 0, 0] # RED for Deleted
        #         deleted_count += 1
        # if deleted_count > 0:
        #     print(f"Slice {i+1}: Removed {deleted_count} vertices.")
        #     sub_mesh_dirty.vertex_colors = o3d.utility.Vector3dVector(colors)
            
        #     # Draw 'Dirty' mesh (with red dots) on top of wireframe
        #     wireframe = o3d.geometry.LineSet.create_from_triangle_mesh(sub_mesh_dirty)
        #     wireframe.paint_uniform_color([0, 0, 0])
            
        #     # To make the red dots obvious, we render them as a PointCloud
        #     pcd_deleted = o3d.geometry.PointCloud()
        #     pcd_deleted.points = o3d.utility.Vector3dVector(dirty_verts[np.all(colors == [1, 0, 0], axis=1)])
        #     pcd_deleted.paint_uniform_color([1, 0, 0])

        #     o3d.visualization.draw_geometries([sub_mesh_dirty, pcd_deleted], 
        #                                     window_name=f"Deleted Items (Red) - Slice {i+1}")
        # else:
        #     print(f"Slice {i+1}: Clean (No vertices removed).")


        tri_labels, num_clusters, _ = sub_mesh.cluster_connected_triangles() #o3d function cluster_connected_triangles
        tri_labels = np.asarray(tri_labels)
        num_new_regions=len(num_clusters)
        
        
        sub_triangles = np.asarray(sub_mesh.triangles) #array of sub-mesh triangles
        sub_vertex_labels = np.zeros(len(sub_mesh.vertices), dtype=int)
    
        sub_vertex_labels[sub_triangles[:, 0]] = tri_labels
        sub_vertex_labels[sub_triangles[:, 1]] = tri_labels
        sub_vertex_labels[sub_triangles[:, 2]] = tri_labels
        safe_count = min(len(mask_indices), len(sub_vertex_labels))
        final_labels[mask_indices[:safe_count]] = region_counter + sub_vertex_labels[:safe_count]

        region_counter += num_new_regions
        #visualize_results(mesh, final_labels)

    return final_labels

def segment_by_centroids_triangles(mesh, saddle_heights, aligned_heights):
    #Uses your existing centroid slicing and clustering logic, but outputs a per-triangle label array for easier smoothing.
    vertices= np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    tri_heights = np.mean(aligned_heights[triangles], axis=1)
    
    cuts = [aligned_heights.min()] + saddle_heights + [aligned_heights.max()]
    cuts = sorted(list(set([round(c, 5) for c in cuts])))
    
    # Initialize region-label array
    global_tri_labels = np.zeros(len(triangles), dtype=int) - 1
    region_counter = 0
    
    print(f"Segmenting into {len(cuts)-1} bands...")
    
    for i in range(len(cuts) - 1):
        lower, upper = cuts[i], cuts[i+1]
        if i == len(cuts) - 2: #Last cut should include end-points
            mask_indices = np.where((tri_heights >= lower - 1e-6) & (tri_heights <= upper + 1e-6))[0]
        else:
            mask_indices = np.where((tri_heights >= lower - 1e-6) & (tri_heights < upper - 1e-6))[0]
    
        if len(mask_indices) == 0: continue
        
        # Build sub-mesh from these triangles
        subset_triangles = triangles[mask_indices]
        unique_v_indices = np.unique(subset_triangles)
        
        global_to_local = {global_id: local_id for local_id, global_id in enumerate(unique_v_indices)}
        vector_map = np.vectorize(global_to_local.get)
        local_triangles = vector_map(subset_triangles)
        
        sub_mesh = o3d.geometry.TriangleMesh()
        sub_mesh.vertices = o3d.utility.Vector3dVector(vertices[unique_v_indices]) # Access via wrapper
        sub_mesh.triangles = o3d.utility.Vector3iVector(local_triangles)
        
        #Visualize
        sub_mesh.paint_uniform_color([1, 0.2, 0.6])
        wireframe = o3d.geometry.LineSet.create_from_triangle_mesh(mesh)
        wireframe.paint_uniform_color([0.8, 0.8, 0.8])
        o3d.visualization.draw_geometries([sub_mesh, wireframe], window_name="Triangle_Centroid_Clusters", mesh_show_back_face=True)
        
        # Cluster connected triangles
        local_labels, cluster_sizes, _ = sub_mesh.cluster_connected_triangles()
        local_labels = np.asarray(local_labels)
        
        # 4. Map Local Cluster IDs directly to Global Triangle IDs; tag the triangles directly.
        for local_t_idx, cluster_id in enumerate(local_labels):
            if cluster_id != -1: # Ignore noise if needed
                global_t_idx = mask_indices[local_t_idx]
                global_tri_labels[global_t_idx] = region_counter + cluster_id
                
        region_counter += len(cluster_sizes)
        
    return global_tri_labels

def minimize_boundary_length(mesh, labels, iterations=10):
    #Relaxes boundary cut to minimize border length (aka sawtooth removal)
    mesh.compute_adjacency_list()
    adj_list=mesh.adjacency_list
    smoothed_labels= labels.copy()
    for iter in range(iterations):
        changes=0
        new_labels= smoothed_labels.copy()
        for idx in range(len(labels)):
            current_lab=smoothed_labels[idx]
            neighbors = adj_list[idx]
            if len(neighbors)==0: continue

            neighbor_labels = [smoothed_labels[n] for n in neighbors]
            counts=np.bincount(neighbor_labels)
            best_label=np.argmax(counts)
            if counts[best_label] > counts[current_lab]:
                new_labels[idx] = best_label
                changes +=1
        smoothed_labels=new_labels

        print(f"{changes} new changes")
        visualize_results(mesh, smoothed_labels)
        if changes == 0: break

    return smoothed_labels

def smooth_triangle_boundaries(tri_labels, tri_adj, iterations=10):
    # Flips triangle labels to straighten jagged boundaries
    colors= color_dual_graph(len(tri_labels), tri_adj)
    labels = tri_labels.copy() #labels of what segment each triangle belongs to
    #indices = list(range(len(labels)))
    
    for i in range(iterations):
        changes = 0
        #np.random.shuffle(indices)
        
        for i in range(2):
            set_indices=np.where(colors==i)[0]
            
            for t in set_indices:

                my_lab = labels[t]
                if my_lab == -1: continue # Skip noise
                
                neighbors = list(tri_adj[t]) #list of specific triangle neighbors
                if not neighbors: continue
                
                votes = [labels[n] for n in neighbors if labels[n] != -1] #look up what segments the neighbors belong to
                if not votes: continue
                
                counts = np.bincount(votes)
                best_label = np.argmax(counts)
                
                if best_label != my_lab and counts[best_label] > len(votes) / 2: #ensures only strict majority flips triangle
                    labels[t] = best_label
                    changes += 1

        print(f"  - Pass {i+1}: Flipped {changes} triangles.")
        visualize_final_triangle_regions(mesh, labels)
        if changes == 0: break

    return labels

def visualize_final_triangle_regions(mesh, tri_labels):
    triangles = np.asarray(mesh.triangles)
    vertices = np.asarray(mesh.vertices)
    unique_ids = np.unique(tri_labels)
    
    submeshes = []
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap("tab20")
    colors = cmap(np.linspace(0, 1, len(unique_ids)))[:, :3]
    
    print(f"Constructing {len(unique_ids)} final region meshes...")
    
    for i, uid in enumerate(unique_ids):
        if uid == -1: continue # Skip noise
        
        # Boolean mask for this region
        mask = (tri_labels == uid)
        subset_tris = triangles[mask]
        
        # Create Mesh Object
        uniq_v, indices = np.unique(subset_tris, return_inverse=True)
        sub = o3d.geometry.TriangleMesh()
        sub.vertices = o3d.utility.Vector3dVector(vertices[uniq_v])
        sub.triangles = o3d.utility.Vector3iVector(indices.reshape(-1, 3))
        
        sub.compute_vertex_normals()
        sub.paint_uniform_color(colors[i % len(colors)])
        submeshes.append(sub)
        
    o3d.visualization.draw_geometries(submeshes, window_name="Final Smoothed Regions")

def get_principal_axis(mesh):
    #Computes the primary axis of the mesh using PCA.
    vertices = np.asarray(mesh.vertices)
    centroid = np.mean(vertices, axis=0)
    centered_verts = vertices - centroid
    cov = np.cov(centered_verts.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    # The last eigenvector corresponds to the largest eigenvalue (max variance)
    principal_axis = eigvecs[:, -1]
    return principal_axis, centroid

def visualize_results(mesh, labels, principal_axis=None, centroid=None):
    #Visualizes the segmented regions
    max_label = labels.max()
    cmap = plt.get_cmap("tab20")
    colors = cmap(np.linspace(0, 1, max_label + 1))[:, :3]
    vertex_colors = np.zeros((len(mesh.vertices), 3))
    for i in range(len(labels)):
        vertex_colors[i] = colors[labels[i]]
    mesh.vertex_colors = o3d.utility.Vector3dVector(vertex_colors)
    o3d.visualization.draw_geometries([mesh], window_name="Morse Decomposition", mesh_show_back_face=True)
    #Adds arrow aligned with principal axis at centroid
    
    # arrow = o3d.geometry.TriangleMesh.create_arrow(
    #     cylinder_radius=0.5, cone_radius=1.0, cylinder_height=10.0, cone_height=3.0
    # )
    # default_axis = np.array([0, 0, 1])
    # if np.allclose(principal_axis, default_axis):
    #     R = np.eye(3)
    # elif np.allclose(principal_axis, -default_axis):
    #     R = -np.eye(3); R[0,0] = 1
    # else:
    #     v = np.cross(default_axis, principal_axis)
    #     c = np.dot(default_axis, principal_axis)
    #     vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    #     R = np.eye(3) + vx + np.dot(vx, vx) * (1 / (1 + c))   
    # arrow.rotate(R, center=(0,0,0))
    # arrow.translate(centroid)
    # arrow.paint_uniform_color([1, 0, 0]) # Red Arrow
    # o3d.visualization.draw_geometries([mesh, arrow], window_name="Morse Decomposition", mesh_show_back_face=True)



if __name__ == "__main__":

    #mesh = o3d.io.read_triangle_mesh("plane_segments\Fuselage_TriHole.stl")
    mesh = o3d.io.read_triangle_mesh("plane_segments\Fuselage_Hole.stl")
    
    #mesh = o3d.geometry.TriangleMesh.create_torus(torus_radius=10.0, tube_radius=3.0, radial_resolution=300, tubular_resolution=150)
    #R = mesh.get_rotation_matrix_from_xyz((np.pi/4, np.pi/4, 0))
    #mesh.rotate(R, center=(0,0,0))
    
    # Pre-processing
    mesh.compute_vertex_normals()
    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()

    axis, center = get_principal_axis(mesh)
    saddle_heights, saddle_indices = find_critical_points(mesh, axis)
    visualize_saddle_points(mesh, saddle_indices)
    aligned_heights = snap_saddle_ridges(mesh, saddle_indices, axis, tolerance=1)
    tri_adj=build_dual_graph(mesh)
    tri_labels=segment_by_centroids_triangles(mesh, saddle_heights, aligned_heights)
    final_labels = smooth_triangle_boundaries(tri_labels, tri_adj)
    visualize_final_triangle_regions(mesh,final_labels)
    #labels = segment_by_saddles(mesh, saddle_heights, aligned_heights)
    #labels=segment_by_centroids(mesh, saddle_heights, aligned_heights)
    #final_labels=minimize_boundary_length(mesh, labels)
    #visualize_results(mesh, final_labels, axis, center)
    


Found 2 saddle points (Splits & Merges).
Visualizing 2 saddle points...
Snapping flat ridges for 2 saddles
Saddle 359 has 67 points affected by snapping
Saddle 1678 has 51 points affected by snapping
Segmenting into 3 bands...
  - Pass 2: Flipped 31 triangles.
Constructing 4 final region meshes...
  - Pass 2: Flipped 0 triangles.
Constructing 4 final region meshes...
Constructing 4 final region meshes...


## Robot-Reachability segmentation

In [ ]:
import pybullet as p
import pybullet_data
import open3d as o3d
import matplotlib.pyplot as plt
import numpy as np
import time
from scipy.spatial import cKDTree
import os
import multiprocessing

ROBOT_PROFILES = {
    "abb_irb120": {
        "elbow_singularity": -1.34,  # -76.9 degrees
        "wrist_singularity": 0.0,
        "shoulder_front_hint": 0.0,  # Reaching Forward (normal)
        "shoulder_back_hint": -2.0,  # Reaching Overhead
        "elbow_up_hint": -1.57,      # Elbow bent up 
        "elbow_down_hint": 0,        # Elbow bend down (normal)
        "wrist_up_hint": -1.57,      # Wrist fipped up
        "wrist_down_hint": 1.57      # Wrist flipped down (normal)
    },

    "ur5": {
        "elbow_singularity": 0.0,    # Straight line
        "wrist_singularity": 0.0,
        "shoulder_front_hint": 0,   # Reaching Forward
        "shoulder_back_hint": 1.57, # Reaching Backwards
        "elbow_up_hint": -1.57,     # Elbow up
        "elbow_down_hint": 1.57,    # Elbow down
        "wrist_up_hint": -1.57,
        "wrist_down_hint": 1.57
    }
}

def worker_process(chunk_indices, chunk_points, chunk_normals, init_kwards, keys, wrist_only):
    init_kwards["connection_mode"] = p.DIRECT
    init_kwards["sample_points"] = False
    local_env = RobotReachability(**init_kwards)
    local_env.generate_all_seeds()
    results=[]
    for i in range(len(chunk_points)):
        pt = chunk_points[i]
        nm = chunk_normals[i]
        res=[]
        for key in keys:
            success, msg = local_env.check_reachability(pt,nm, key, wrist_only)
            res.append(success)
        results.append((chunk_indices[i], tuple(res)))

    p.disconnect(local_env.client)
    return results

class RobotReachability:
    
    def __init__(self, num_points, urdf_path, mesh_path, mesh_position=[0,0.2,0], 
                 mesh_orientation=[0,0,0], base_position=[0,0,0], robot_name="abb_irb120",
                 connection_mode=p.GUI, sample_points=True, shared_mesh_path=None):
        
        #save kwagrs for worker processes
        self.urdf_path=urdf_path
        self.mesh_path=mesh_path
        self.mesh_position=mesh_position
        self.mesh_orientation=mesh_orientation
        self.base_position=base_position
        self.robot_name=robot_name
        self.shared_mesh_path = shared_mesh_path
        
        # Load robotProfile
        if robot_name not in ROBOT_PROFILES:
            raise ValueError(f"Unsupported robot '{robot_name}'. Available profiles: {list(ROBOT_PROFILES.keys())}")
        self.profile = ROBOT_PROFILES[robot_name]  
        print("Loaded profile for", robot_name)    

        #Setup PyBullet
        self.client=p.connect(connection_mode)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0,0,-9.8)
        
        #Load Robot
        self.robot_id = p.loadURDF(urdf_path, basePosition=base_position, useFixedBase=True)
        self.num_joints = p.getNumJoints(self.robot_id)        
        self.ee_index= -1
        self.movable_joints = [] 
        #print(f"Total Joints Found: {self.num_joints}")
        for i in range(self.num_joints):
            info = p.getJointInfo(self.robot_id, i)
            joint_name = info[1].decode("utf-8")
            link_name = info[12].decode("utf-8")
            joint_type = info[2]
            # Identify End Effector by name (Usually 'link_6' or 'tool0')
            if connection_mode == p.GUI:
                print(f"ID {i}: Joint='{joint_name}' Link='{link_name}' Type={joint_type}")
            if "link_6" in link_name or "tool0" in link_name:
                self.ee_index = i
            # Store movable joints for IK control
            if joint_type != p.JOINT_FIXED:
                self.movable_joints.append(i)

        if self.ee_index == -1:
            print("WARNING: Could not find 'link_6'. Defaulting to last index.")
            self.ee_index = self.num_joints - 1

        if sample_points:
            #Mesh Processing
            self.mesh=o3d.io.read_triangle_mesh(mesh_path)
            if not self.mesh.has_triangles():
                raise ValueError("CRITICAL ERROR: Failed to load mesh. File path may be wrong.")
            self.mesh.remove_duplicated_vertices()
            self.mesh.remove_duplicated_triangles()
            self.mesh.remove_degenerate_triangles()
            self.mesh.compute_vertex_normals()
            self.mesh = self.mesh.subdivide_midpoint(number_of_iterations=3)

            self.mesh.scale(0.001, center=(0,0,0))
            center_offset = self.mesh.get_center()
            self.mesh.translate(-center_offset)

            #save file once for all parallel processes to load
            self.shared_mesh_path = os.path.abspath("shared_temp_mesh.stl")
            o3d.io.write_triangle_mesh(self.shared_mesh_path, self.mesh)

            #Mesh Positioning and Sampling
            mesh_quat = p.getQuaternionFromEuler(mesh_orientation)
            rot_matrix_flat=p.getMatrixFromQuaternion(mesh_quat)
            R=np.array(rot_matrix_flat).reshape(3,3)
            self.mesh.rotate(R, center=(0, 0, 0))
            self.mesh.translate(mesh_position)

            self.pcd=self.mesh.sample_points_poisson_disk(number_of_points=num_points)
            self.points = np.asarray(self.pcd.points)
            self.normals= np.asarray(self.pcd.normals)
            print(f"Loaded mesh with with {len(self.points)} subsampled points")
            self.max_reach= self.calculate_max_reach()
            self.reachable_mask = self.filter_unreachable_by_distance(self.max_reach)

        else:
            self.mesh=None
            self.points=[]
            self.normals=[]
            self.max_reach=None
            self.reachable_mask=None

        #Load Mesh into PyBullet
        mesh_quat = p.getQuaternionFromEuler(mesh_orientation)
        visual_id = p.createVisualShape(shapeType = p.GEOM_MESH, fileName=self.shared_mesh_path, rgbaColor=[0.6, 0.6, 0.6, 1], meshScale=[1,1,1])
        collision_id = p.createCollisionShape(shapeType = p.GEOM_MESH, fileName = self.shared_mesh_path, meshScale=[1,1,1], flags=p.GEOM_FORCE_CONCAVE_TRIMESH)
        
        self.mesh_body_id = p.createMultiBody(
            baseMass=0,
            baseCollisionShapeIndex=collision_id,
            baseVisualShapeIndex=visual_id,
            basePosition=mesh_position,
            baseOrientation=mesh_quat
        )

        #Storage for Cell Divisions
        self.signatures=[]
        self.cell_ids=[]
    
    def generate_all_seeds(self):
        #Not super generalized
        self.seeds={}
        prof=self.profile
        shoulder_opts=["front", "back"]
        elbow_opts= ['elbow_up', 'elbow_down']
        wrist_opts= ['wrist_up', "wrist_down"]
        for s in shoulder_opts:
            for e in elbow_opts:
                for w in wrist_opts:
                    name=f"{s}_{e}_{w}"
                    j1= prof["shoulder_front_hint"] if s =="front" else prof["shoulder_back_hint"]
                    j3= prof["elbow_up_hint"] if e == "elbow_up" else prof["elbow_down_hint"]
                    j5= prof["wrist_up_hint"] if w == "wrist_up" else prof["wrist_down_hint"]
                    j2 = 0.0 if s=="front" else -1.57
                    self.seeds[name]=[j1, j2, j3, 0, j5, 0]
        print(f"Generated {len(self.seeds)} seeds for {len(shoulder_opts)*len(elbow_opts)*len(wrist_opts)} regions.")

    def align_vector_to_normal(self,normal):
        #Calculate Quaternion to align robot z-axis to tool
        tool_axis=np.array([0,0,1])
        target_axis = normal
        rotation_axis=np.cross(tool_axis,target_axis)
        if np.linalg.norm(rotation_axis)<1e-6: #already aligned
            if np.dot(tool_axis, target_axis) > 0: 
                return [0, 0, 0, 1]
            return [0,0,0,1] 
                
        rotation_axis= rotation_axis/np.linalg.norm(rotation_axis)
        angle = np.arccos(np.dot(tool_axis, target_axis)) 
        q_xyz=rotation_axis*np.sin(angle/2)
        q_w=np.cos(angle/2)
        return list(q_xyz) + [q_w]   

    def calculate_max_reach(self):
        #Calculating maximum reach from URDF (summing joint distances) - could add saftey factor value
        current_joint_index = self.ee_index
        total_length=0.0
        while current_joint_index !=-1:
            joint_info = p.getJointInfo(self.robot_id, current_joint_index)
            parent_link_index=joint_info[16]
            offset_vec= joint_info[14]
            dist=np.linalg.norm(offset_vec)
            #print(f"Link {current_joint_index} is child of Link {parent_link_index}, and is offset by {dist}")
            total_length+=dist
            current_joint_index=parent_link_index  
        #print(f"Calculated Chain Length: {total_length:.3f}m")
        safety_factor = 1.5
        return(total_length*safety_factor)
    
    def filter_unreachable_by_distance(self, max_reach):
        dists=np.linalg.norm(self.points, axis=1)
        mask = dists < (max_reach + 0.1)
        print(f"Distance Culling: {len(self.points)-np.sum(mask)} points skipped due to being completely unreachable")
        return mask
    
    def check_manipulability(self, joint_positions): #TODO: Need to integrate!
        #Calculate Yoshikawa manipulability Index, lower values indicate proximity to singularity
        jac_t, jac_r = p.calculateJacobian(
            self.robot_id, 
            self.ee_index, 
            localPosition=[0,0,0], 
            objPositions=joint_positions, 
            objVelocities=[0]*len(joint_positions), 
            objAccelerations=[0]*len(joint_positions)
        )
        jacobian = np.vstack((jac_t, jac_r))
        manipulability = np.sqrt(np.linalg.det(jacobian @ jacobian.T))
        return manipulability

    def check_collision(self):
        #Return true if colliding with self or environment
        contact_points=p.getContactPoints(bodyA=self.robot_id, bodyB=self.robot_id)
        for contact in contact_points:
            linkA, linkB = contact[3], contact[4]
            if abs(linkA-linkB) > 1 and contact[8] < -0.005: #Eliminates same-link overlap as false positive
                return True, "Self Collision"
        env_contacts = p.getContactPoints(bodyA=self.robot_id, bodyB=self.mesh_body_id)
        for contact in env_contacts:
            distance = contact[8]
            if distance < -0.005: 
                return True, "Part Collision"
        return False, "No Collision"
    
    def check_reachability(self, target_pos, target_normal, seed_name, wrist_only=False):
        prof=self.profile
        #Set up joint cage
        ll, ul=[], [] #lower/upper limit
        jr, rp=[], [] #joint range, rest pose
        is_front= "front" in seed_name
        is_elbow_up= "elbow_up" in seed_name
        is_wrist_up= "wrist_up" in seed_name
        
        seed_conf=self.seeds[seed_name]
        target_azimuth = np.arctan2(target_pos[1], target_pos[0]) #Calculate the angle in the XY plane for shoulder hinting

        if wrist_only:
            new_seed_conf= [0]*len(self.movable_joints)
            new_seed_conf[4] = seed_conf[4] #Only set wrist joint,
            seed_conf=new_seed_conf

        singularity_buffer= 0.05
        for i, joint_id in enumerate(self.movable_joints):
            info=p.getJointInfo(self.robot_id, joint_id)
            lower, upper=info[8], info[9]
            current_rest= seed_conf[i]
            #Wrist Joint limits (Assumed J5, split usually at 0 deg- Our convention: wrist up means J5 < split)
            if joint_id==4:
                split = prof["wrist_singularity"]
                if is_wrist_up: 
                    upper = split - singularity_buffer
                else:
                    lower = split + singularity_buffer
            if not wrist_only: #Only assign check elbow/sholder joint limits for full seeds

                #Elbow Joint limits (Assumed J3, split varies - Our convention: wrist up means J5 < split)
                if joint_id==2:
                    split = prof["elbow_singularity"]
                    if is_elbow_up: 
                        upper = -singularity_buffer + split
                    else:
                        lower = singularity_buffer + split

                #Shoulder Joint rests (Assumed J1, rest depends on target azimuth)
                if joint_id==0:
                    if is_front:
                        current_rest=target_azimuth
                    else:
                        current_rest=(target_azimuth%(2*np.pi)-np.pi) #flip to opposite side of the circle

                    #For ABB, joint limit is actually +- 165 degrees, this ensure rest pose respects limits
                    if current_rest > upper:
                        current_rest= upper-0.1
                    if current_rest < lower:
                        current_rest= lower+0.1

            ll.append(lower)
            ul.append(upper)
            jr.append(upper-lower)
            rp.append(current_rest)

        for i, joint_id in enumerate(self.movable_joints):
             p.resetJointState(self.robot_id, joint_id, seed_conf[i])

        target_orient = self.align_vector_to_normal(-target_normal)

        # 3. Run IK with the Limits (The Cage)
        joint_poses = p.calculateInverseKinematics(
            self.robot_id,
            self.ee_index,
            target_pos,
            target_orient,
            lowerLimits=ll,    # <--- Enforces all joint limits
            upperLimits=ul,    # <--- Enforces all joint limits
            jointRanges=jr,
            restPoses=rp,
            maxNumIterations=200, 
            residualThreshold=1e-3
        )

        all_joint_positions=[]
        for i, joint_id in enumerate(self.movable_joints):
            p.resetJointState(self.robot_id, joint_id, joint_poses[i])
            all_joint_positions.append(joint_poses[i])

        #quality=self.check_manipulability(all_joint_positions)
        actual_pos = p.getLinkState(self.robot_id, self.ee_index)[4]
        dist = np.linalg.norm(np.array(actual_pos) - np.array(target_pos))
        if dist > 0.01: 
            return False, f"Unreachable in {seed_name} ({dist:.3f}m)"
        is_collision, _ = self.check_collision()
        if is_collision:
            return False, "Collision"
        return True, "Success"

    def run_parallel_analysis(self, wrist_only=False, num_processes=None):
        if num_processes is None:
            num_processes= max(1, multiprocessing.cpu_count() - 1)
        self.generate_all_seeds()
        keys=list(self.seeds.keys())
        if wrist_only:
            keys=keys[:2]
        print(f"Running parallel analysis with {num_processes} processes with orientations: {keys}")
        feasible_mask=self.reachable_mask
        valid_indices=np.where(feasible_mask)[0]
        valid_points=self.points[valid_indices]
        valid_normals=self.normals[valid_indices]
        
        #Shuffle data
        suffle_idx=np.random.permutation(len(valid_indices))
        valid_indices=valid_indices[suffle_idx]
        valid_points=valid_points[suffle_idx]
        valid_normals=valid_normals[suffle_idx]

        #Chunk Data
        chunks_idx = np.array_split(valid_indices, num_processes)
        chunks_points = np.array_split(valid_points, num_processes)
        chunks_normals = np.array_split(valid_normals, num_processes)

        init_kwargs={ #Generic kwargs to initialize worker processes
            'num_points': 0,
            'urdf_path': self.urdf_path,
            'mesh_path': self.mesh_path,
            'mesh_position': self.mesh_position,
            'mesh_orientation': self.mesh_orientation,
            'base_position': self.base_position,
            'robot_name': self.robot_name
        }
        
        tasks=[]
        for i in range(num_processes): #load up tasks
            if len(chunks_idx[i])>0:
                tasks.append((chunks_idx[i], chunks_points[i], chunks_normals[i], init_kwargs, keys, wrist_only))
        print(f"Starting parallel analysis on {len(valid_indices)} points, using {num_processes} processes")
        start_time=time.time()
        with multiprocessing.Pool(processes=num_processes) as pool: #distribute tasks  
            results= pool.starmap(worker_process, tasks)
            
        self.signatures = [tuple([False]*len(keys))]*len(self.points)
        for worker_res in results:
            for original_idx,sig in worker_res:
                self.signatures[original_idx]=sig

        print(f"Parallel analysis complete in {time.time()-start_time:.2f} seconds")

    def run_analysis(self, wrist_only=False):
        self.generate_all_seeds()
        feasible_mask = self.reachable_mask
        self.signatures=[]
        keys = list(self.seeds.keys())
        if wrist_only: #Only Keep 1 wrist up and 1 wrist down seed, and don't check elbow/sholder joint limits 
            keys = keys[:2] 
        print(keys)

        #print(f"Starting analysis on {len(self.points)} points")
        start_time=time.time()
        for i in range(len(self.points)):
            if not feasible_mask[i]:
                self.signatures.append(tuple([False]*len(keys)))
                continue
            pt = self.points[i]
            nm = self.normals[i]
            res=[]
            for key in keys:
                result, msg = self.check_reachability(pt, nm, key, wrist_only)
                res.append(result)
            self.signatures.append(tuple(res))

            #if i%200==0: print(f" Processed {i}/{len(self.points)}")
        print(f"Analysis complete in {time.time()-start_time:.2f} seconds")

    def visualize_signatures(self):
        #Colors points strictly by what they can do - wrist only.
        colors = []
        for sig in self.signatures:
            up, down = sig
            if up and down:
                colors.append([0, 1, 0]) # Green (Both)
            elif up and not down:
                colors.append([1, 1, 0]) # Yellow (Up Only)
            elif down and not up:
                colors.append([0, 0, 1]) # Blue (Down Only)
            else:
                colors.append([1, 0, 0]) # Red (Neither)
        self.pcd.colors = o3d.utility.Vector3dVector(np.array(colors))
        frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
        mesh_wire = o3d.geometry.LineSet.create_from_triangle_mesh(self.mesh)
        o3d.visualization.draw_geometries([self.pcd, mesh_wire, frame])  

    def segment_into_cells(self, radius=0.02):
        #print("Segmeneting surface into cells")
        tree= cKDTree(self.points)
        self.cell_ids = -1 * np.ones(len(self.points), dtype = int)
        current_cell_id=0
        for i in range(len(self.points)):
            if self.cell_ids[i] != -1: continue
            current_sig=self.signatures[i]
            queue=[i]
            self.cell_ids[i] = current_cell_id
            while len(queue)>0:
                idx=queue.pop(0)
                neighbors=tree.query_ball_point(self.points[idx], r=radius)
                for n_idx in neighbors:
                    if self.cell_ids[n_idx] == -1:
                        if self.signatures[n_idx] == current_sig:
                            self.cell_ids[n_idx]=current_cell_id
                            queue.append(n_idx)

            current_cell_id +=1
        print(f"Segmentation complete. Found {current_cell_id+1} unique cells")

    def map_cells_to_mesh(self, distance_threshold=0.05):
        mesh_vertices = np.asarray(self.mesh.vertices)
        valid_mask= self.cell_ids != -1
        valid_points = self.points[valid_mask] #Only consider points that were assigned to a reachable cell
        valid_cell_ids= self.cell_ids[valid_mask]
        self.vertex_cell_ids = -1 *np.ones(len(mesh_vertices), dtype=int) #All vertices start as unassigned
        if len(valid_points)==0: 
            print("No valid points to map to mesh")
            return
        #Build KDTree with reachable points
        tree = cKDTree(valid_points)
        distances, nearest_indices = tree.query(mesh_vertices) #return nearest point for each vertex

        valid_mapping_mask = distances < distance_threshold #Only assign if vertex is close enough to a valid point
        self.vertex_cell_ids[valid_mapping_mask] = valid_cell_ids[nearest_indices[valid_mapping_mask]]
        mapped_count = np.sum(valid_mapping_mask)
        print(f"Mapped {mapped_count} out of {len(mesh_vertices)} vertices to {len(np.unique(valid_cell_ids))} unique cells.")

    def visualize_result(self):
        num_cells=self.cell_ids.max() +1
        cell_colors_map = np.random.uniform(0,1, (num_cells,3))
        final_colors=[]
        for i in range((len(self.points))):
            if self.signatures[i] == (False,False):
                final_colors.append([1,0,0])
            else:
                c_id=self.cell_ids[i]
                final_colors.append(cell_colors_map[c_id])
        self.pcd.colors=o3d.utility.Vector3dVector(np.array(final_colors))
        mesh_wire=o3d.geometry.LineSet.create_from_triangle_mesh(self.mesh)
        o3d.visualization.draw_geometries([self.pcd, mesh_wire])

    def visualize_with_plotly(self, filename="reachability_map.html"):
        import plotly.graph_objects as go
        import plotly.colors as pc
        fig=go.Figure()
        keys=list(self.seeds.keys())
        v=np.asarray(self.mesh.vertices)
        t=np.asarray(self.mesh.triangles)
        fig.add_trace(go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=t[:,0], j=t[:,1], k=t[:,2], color='lightgrey', opacity=0.5))
        #colors = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#00FFFF', '#FF00FF', '#FFA500', '#800080']
        unique_cells = np.unique(self.cell_ids)
        keys= list(self.seeds.keys())
        palette=pc.sample_colorscale("Turbo", len(unique_cells))

        for i, cell_id in enumerate(unique_cells):
            if cell_id == -1:
                continue
            indices = np.where(self.cell_ids == cell_id)[0]
            pts = self.points[indices]
            sig= self.signatures[indices[0]]
            valid_configs=[keys[k] for k, is_valid in enumerate(sig) if is_valid]
            # Format the hover text
            # We use <br> to break lines in the hover box
            hover_text = f"<b>Cell ID: {cell_id}</b><br>Points: {len(pts)}<br><br><b>Valid Configs:</b><br>" + "<br>".join(valid_configs)
            
            fig.add_trace(go.Scatter3d(
                x=pts[:,0], 
                y=pts[:,1], 
                z=pts[:,2],
                mode='markers',
                marker=dict(
                    size=4, 
                    color=palette[i], # Assign unique color
                    line=dict(width=0)
                ),
                name=f"Cell {cell_id}",
                text=[hover_text] * len(pts), # Apply text to all points
                hoverinfo='text' # Only show our custom text
            ))
            
        fig.update_layout(
            title=f"Reachability Segmentation ({len(unique_cells)} Unique Regions)",
            scene=dict(aspectmode='data'),
            legend_title_text='Segments'
        )
        
        fig.write_html(filename)

    def visualize_solid_mesh(self):
            print("Visualizing solid mapped mesh...")
            num_cells = np.max(self.cell_ids) + 1
            np.random.seed(42) 
            cell_colors_map = np.random.uniform(0.1, 0.9, (num_cells, 3))
            vertex_colors = np.zeros((len(self.vertex_cell_ids), 3))
            for i, v_id in enumerate(self.vertex_cell_ids):
                if v_id == -1:
                    vertex_colors[i] = [1, 0, 0] # Red for unreachable
                else:
                    vertex_colors[i] = cell_colors_map[v_id]
            self.mesh.vertex_colors = o3d.utility.Vector3dVector(vertex_colors)           
            o3d.visualization.draw_geometries([self.mesh], mesh_show_back_face=True)

if __name__ == "__main__":
    multiprocessing.freeze_support()

    abb_urdf_file = r"C:\Users\jonas\BARC_NDI\pointcloudcpp\abbIrb120.urdf" 
    ur_urdf_file=r"C:\Users\jonas\BARC_NDI\pointcloudcpp\ur5.urdf"
    mesh_file = r"C:\Users\jonas\BARC_NDI\pointcloudcpp\plane_segments\Airfoil_Surface_example.stl"
    
    # Placement: 0.5m forward, 0.2m up. Rotated upright.
    part_pos = [0.5, 0, 0.3] 
    part_euler = [0, 1.57, 3.14] 
    num_points= 5000

    segmenter = RobotReachability(num_points, abb_urdf_file, mesh_file, mesh_position=part_pos, mesh_orientation=part_euler)
    
    # Reachability Check
    wrist_only=False
    #segmenter.run_analysis(wrist_only)
    segmenter.run_parallel_analysis(wrist_only)
    if wrist_only:
        segmenter.visualize_signatures()


    # Cell Formation (Radius = 5cm neighbor search)
    segmenter.segment_into_cells(radius=0.05)
    segmenter.map_cells_to_mesh(distance_threshold=0.05)
    
    # 3. View Results
    segmenter.visualize_result()
    segmenter.visualize_with_plotly("my_robot_cells.html")
    segmenter.visualize_solid_mesh()
    if p.isConnected():
        p.disconnect()



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loaded profile for abb_irb120
ID 0: Joint='joint_1' Link='link_1' Type=0
ID 1: Joint='joint_2' Link='link_2' Type=0
ID 2: Joint='joint_3' Link='link_3' Type=0
ID 3: Joint='joint_4' Link='link_4' Type=0
ID 4: Joint='joint_5' Link='link_5' Type=0
ID 5: Joint='joint_6' Link='link_6' Type=0
ID 6: Joint='joint6-tool0' Link='tool0' Type=4
ID 7: Joint='base_link-base' Link='base' Type=4
Loaded mesh with with 5000 subsampled points
Distance Culling: 2526 points skipped due to being completely unreachable
Generated 8 seeds for 8 regions.
Running parallel analysis with 15 processes with orientations: ['front_elbow_up_wrist_up', 'front_elbow_up_wrist_down', 'front_elbow_down_wrist_up', 'front_elbow_down_wrist_down', 'back_elbow_up_wrist_up', 'back_elbow_up_wrist_down', 'back_elbow_down_wrist_up', 'back_elbow_down_wrist_down']
Star

In [8]:
import open3d as o3d


mesh_file = r"C:\Users\jonas\NDIBARC\pointcloudcpp\plane_segments\Airfoil_Surface_example.stl"
mesh=o3d.io.read_triangle_mesh(mesh_file)
mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles()
mesh.compute_vertex_normals()

pcd = o3d.geometry.PointCloud()
pcd.points = mesh.vertices
normals=o3d.utility.Vector3dVector(-np.asarray(mesh.vertex_normals))
pcd.normals = normals
o3d.visualization.draw_geometries([mesh,pcd], point_show_normal=True, mesh_show_back_face=True)


[Open3D WARNING] Unable to load file C:\Users\jonas\NDIBARC\pointcloudcpp\plane_segments\Airfoil_Surface_example.stl with ASSIMP: Unable to open file "C:\Users\jonas\NDIBARC\pointcloudcpp\plane_segments\Airfoil_Surface_example.stl".
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.


In [ ]:
import multiprocessing
print(multiprocessing.cpu_count())


16
